### 1. Preparação do Ambiente e Credenciais
> Nesta etapa inicial, o código localiza e carrega o arquivo `.env`. A decisão arquitetural de utilizar a biblioteca `python-dotenv` garante que as credenciais de acesso ao banco de dados não fiquem expostas (hardcoded) no código-fonte, seguindo as diretrizes rigorosas de segurança da informação e engenharia de software.

In [0]:
import os
from dotenv import load_dotenv

# LOCALIZAÇÃO E CARGA DO ARQUIVO .ENV
caminho_env = None
for tentativa in [".env", "../.env", "../../.env"]:
    if os.path.exists(tentativa):
        caminho_env = tentativa
        break

if caminho_env:
    load_dotenv(dotenv_path=caminho_env)
    print(f"✅ Arquivo .env localizado e carregado com sucesso pelo sistema a partir de: '{caminho_env}'")
else:
    raise FileNotFoundError("⚠️ ERRO CRÍTICO: O sistema não localizou o arquivo .env.")

### 2. Conexão e Extração de Dados (SQL Server -> Databricks)
> Para realizar a leitura, o pipeline utiliza o conector nativo do Databricks para SQL Server (`.format("sqlserver")`). Esta abordagem atende aos requisitos de segurança do ambiente Serverless e permite extrair os dados da tabela `squad2.ecommerce_pedidos` diretamente do banco de dados relacional para um DataFrame PySpark na memória do cluster.

In [0]:
# RESGATE DE CREDENCIAIS
jdbc_host = os.getenv("SQL_HOST")
jdbc_db = os.getenv("SQL_DATABASE")
jdbc_user = os.getenv("SQL_USERNAME")
jdbc_pass = os.getenv("SQL_PASSWORD")

nome_tabela = "squad2.ecommerce_pedidos"
print(f"📥 O sistema está iniciando a leitura dos dados oficiais da tabela '{nome_tabela}' no SQL Server...")

# EXTRAÇÃO VIA CONECTOR NATIVO
df_analise = (
    spark.read
    .format("sqlserver")
    .option("host", jdbc_host)
    .option("port", "1433")
    .option("database", jdbc_db)
    .option("user", jdbc_user)
    .option("password", jdbc_pass)
    .option("dbtable", nome_tabela)
    .option("encrypt", "true")
    .option("trustServerCertificate", "false")
    .load()
)

print("✅ Tabela carregada com sucesso para a memória do cluster.")

### 3. Inspeção Visual e Validação de Estrutura
> Aqui ocorre a primeira validação visual dos dados consumidos. O comando `printSchema()` exibe os tipos de dados estruturais inferidos para cada coluna (ex: inteiros, strings). Em seguida, a função nativa `display()` renderiza uma amostra interativa das primeiras 10 linhas da tabela, permitindo auditar visualmente se o formato dos dados corresponde ao esperado após a ingestão.

In [0]:
print("📊 Estrutura Técnica da Tabela (Schema):")
df_analise.printSchema()

print("\n🔍 Renderizando amostra dos 10 primeiros registros na interface visual:")
# A função display é a responsável por desenhar a tabela estilizada na tela do Databricks
display(df_analise.limit(10))

### 4. Validação de Qualidade de Dados (Data Quality)
> Esta etapa executa uma checagem de integridade analítica. O algoritmo calcula o volume total de registros e colunas, além de realizar uma auditoria na coluna primária (`id_pedido`). O objetivo é garantir a ausência de valores nulos na chave identificadora, cenário que indicaria falhas graves na consistência dos dados ingeridos na etapa anterior.

In [0]:
from pyspark.sql.functions import col

print("⚙️ Processando métricas de qualidade de dados...\n")

# CONTAGEM DE VOLUME
total_linhas = df_analise.count()
total_colunas = len(df_analise.columns)

print("📈 VOLUME DE DADOS:")
print(f"Total de Registros: {total_linhas}")
print(f"Total de Colunas: {total_colunas}")

# AUDITORIA DA CHAVE PRIMÁRIA
chave_primaria = "id_pedido"

if chave_primaria in df_analise.columns:
    qtd_nulos_chave = df_analise.filter(col(chave_primaria).isNull()).count()
    
    print(f"\n🛡️ SAÚDE DA CHAVE PRIMÁRIA ('{chave_primaria}'):")
    if qtd_nulos_chave == 0:
        print("✅ Status: Saudável. Não foi detectado nenhum valor nulo na chave primária.")
    else:
        print(f"⚠️ Status: Crítico. O sistema detectou {qtd_nulos_chave} registros com chave nula.")
else:
    print(f"\n⚠️ Aviso: A coluna identificadora '{chave_primaria}' não foi localizada no schema atual.")